In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
SILVER_TABLE = (
    "weather_streaming.weather.weather_silver"
)

GOLD_DAILY_TABLE = (
    "weather_streaming.weather.weather_gold_daily"
)

GOLD_LATEST_TABLE = (
    "weather_streaming.weather.weather_gold_latest"
)

In [0]:
silver_df = (
    spark.table(SILVER_TABLE)
)

In [0]:
display(
    silver_df
    .orderBy(
        F.col("event_time").desc()
    )
    .limit(20)
)

In [0]:
silver_df = (
    silver_df
    .withColumn(
        "observation_date",
        F.to_date(
            F.col("event_time")
        )
    )
)

In [0]:
gold_daily_df = silver_df.groupBy("city", "observation_date").agg(
    F.avg("temperature_c").alias("avg_temperature_c"),
    F.min("temperature_c").alias("min_temperature_c"),
    F.max("temperature_c").alias("max_temperature_c"),
    F.avg("humidity_pct").alias("avg_humidity_pct"),
    F.max("wind_speed_kmh").alias("max_wind_speed_kmh"),
    F.sum("precipitation_mm").alias("total_precipitation_mm"),
    F.count("*").alias("observation_count"),
)

In [0]:
gold_daily_df = (
    gold_daily_df.withColumn(
        "avg_temperature_c", F.round(F.col("avg_temperature_c"), 2)
    )
    .withColumn("avg_humidity_pct", F.round(F.col("avg_humidity_pct"), 2))
    .withColumn("max_wind_speed_kmh", F.round(F.col("max_wind_speed_kmh"), 2))
    .withColumn("total_precipitation_mm", F.round(F.col("total_precipitation_mm"), 2))
)

In [0]:
gold_daily_df = gold_daily_df.withColumn(
    "temperature_category",
    F.when(F.col("avg_temperature_c") < 15, "COLD")
    .when(F.col("avg_temperature_c") < 25, "MODERATE")
    .when(F.col("avg_temperature_c") < 35, "WARM")
    .otherwise("HOT"),
)

In [0]:
gold_daily_df = gold_daily_df.withColumn(
    "rain_status",
    F.when(F.col("total_precipitation_mm") > 0, "RAIN").otherwise("NO_RAIN"),
)

In [0]:
gold_daily_df = gold_daily_df.withColumn("gold_processed_at", F.current_timestamp())

In [0]:
gold_daily_df = gold_daily_df.select(
    "city",
    "observation_date",
    "avg_temperature_c",
    "min_temperature_c",
    "max_temperature_c",
    "temperature_category",
    "avg_humidity_pct",
    "max_wind_speed_kmh",
    "total_precipitation_mm",
    "rain_status",
    "observation_count",
    "gold_processed_at",
)

In [0]:
invalid_gold = gold_daily_df.filter(
    F.col("city").isNull() | F.col("observation_date").isNull()
)

invalid_count = invalid_gold.count()

if invalid_count > 0:
    raise ValueError(f"Gold validation failed. Invalid records: {invalid_count}")

In [0]:
(
    gold_daily_df.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(GOLD_DAILY_TABLE)
)

In [0]:
gold_daily_df.createOrReplaceTempView("gold_daily_source")

In [0]:
spark.sql(
    f"""
    MERGE INTO {GOLD_DAILY_TABLE} AS target

    USING gold_daily_source AS source

    ON
        target.city = source.city
        AND target.observation_date =
            source.observation_date

    WHEN MATCHED THEN UPDATE SET
        target.avg_temperature_c =
            source.avg_temperature_c,

        target.min_temperature_c =
            source.min_temperature_c,

        target.max_temperature_c =
            source.max_temperature_c,

        target.temperature_category =
            source.temperature_category,

        target.avg_humidity_pct =
            source.avg_humidity_pct,

        target.max_wind_speed_kmh =
            source.max_wind_speed_kmh,

        target.total_precipitation_mm =
            source.total_precipitation_mm,

        target.rain_status =
            source.rain_status,

        target.observation_count =
            source.observation_count,

        target.gold_processed_at =
            source.gold_processed_at

    WHEN NOT MATCHED THEN INSERT *
    """
)

In [0]:
latest_window = Window.partitionBy("city").orderBy(
    F.col("event_time").desc(), F.col("ingestion_time").desc()
)

In [0]:
gold_latest_df = (
    silver_df.withColumn("row_number", F.row_number().over(latest_window))
    .filter(F.col("row_number") == 1)
    .drop("row_number")
)

In [0]:
gold_latest_df = gold_latest_df.withColumn("gold_processed_at", F.current_timestamp())

In [0]:
gold_latest_df = gold_latest_df.select(
    "city",
    "event_time",
    "latitude",
    "longitude",
    "temperature_c",
    "temperature_category",
    "humidity_pct",
    "wind_speed_kmh",
    "precipitation_mm",
    "ingestion_time",
    "source",
    "gold_processed_at",
)

In [0]:
(
    gold_latest_df.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(GOLD_LATEST_TABLE)
)